In [ ]:
import sys
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# permite importar desde src/ (ajusta si tu notebook esta en otra profundidad)
sys.path.append(str(Path.cwd().parent))

from src.segmentation.edge_detection import sobel_gauss_otsu, bordes_sv, densidad_bordes
from src.segmentation.morphological_ops import rellenar_contornos

PROCESSED_DIR = Path('../data/processed/train')


In [ ]:
clase_analizar = 'organic'
N_IMGS = 25
imagenes = sorted((PROCESSED_DIR / clase_analizar).iterdir())[:N_IMGS]

UMBRAL_DENSIDAD = 0.15
K_GAUSS = 11

for img_path in imagenes:
    img_bgr = cv2.imread(str(img_path))
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    hsv_raw = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    S_raw = cv2.GaussianBlur(hsv_raw[:, :, 1], (5, 5), 0)
    densidad = densidad_bordes(S_raw, K_GAUSS)

    if densidad > UMBRAL_DENSIDAD:
        img_lp = cv2.bilateralFilter(img_bgr, d=50, sigmaColor=220, sigmaSpace=300)
    else:
        img_lp = img_bgr.copy()

    img_hsv = cv2.cvtColor(img_lp, cv2.COLOR_BGR2HSV)
    H, S, V = cv2.split(img_hsv)

    sobel_comb = bordes_sv(S, V, K_GAUSS)
    abierto, cerrado, relleno, k_open, k_close = rellenar_contornos(sobel_comb)
    objeto = cv2.bitwise_and(img_rgb, img_rgb, mask=relleno)

    # ---- metricas ----
    area_obj = np.count_nonzero(relleno)
    cobertura = 100 * area_obj / relleno.size
    n_comp = cv2.connectedComponentsWithStats(relleno, 8)[0]
    n_objetos = max(0, n_comp - 1)
    dens_bordes = 100 * np.count_nonzero(sobel_comb) / sobel_comb.size

    overlay = img_rgb.copy()
    overlay[relleno > 0] = (0.5 * overlay[relleno > 0] + 0.5 * np.array([255, 0, 0])).astype(np.uint8)

    # ---- visualizacion ----
    fig, axes = plt.subplots(2, 4, figsize=(22, 11))
    plt.rcParams['figure.dpi'] = 150

    axes[0, 0].imshow(img_rgb)
    axes[0, 0].set_title('Original', fontsize=10)
    axes[0, 0].axis('off')
    axes[0, 1].imshow(sobel_comb, cmap='gray')
    axes[0, 1].set_title(f'Bordes Sobel S|V (dens={dens_bordes:.1f}%)', fontsize=10)
    axes[0, 1].axis('off')
    axes[0, 2].imshow(cerrado, cmap='gray')
    axes[0, 2].set_title(f'Closing (k_open={k_open}, k_close={k_close})', fontsize=10)
    axes[0, 2].axis('off')
    axes[0, 3].imshow(relleno, cmap='gray')
    axes[0, 3].set_title(f'Relleno ({n_objetos} obj)', fontsize=10)
    axes[0, 3].axis('off')

    axes[1, 0].imshow(objeto)
    axes[1, 0].set_title('Objeto segmentado', fontsize=10)
    axes[1, 0].axis('off')
    axes[1, 1].imshow(overlay)
    axes[1, 1].set_title(f'Overlay ({cobertura:.1f}%)', fontsize=10)
    axes[1, 1].axis('off')

    axes[1, 2].axis('off')
    texto = (
        f"MÉTRICAS\n"
        f"------------------\n"
        f"Densidad S inicial: {densidad:.3f}\n"
        f"Filtro pasabajo: {'SÍ' if densidad > UMBRAL_DENSIDAD else 'NO'}\n"
        f"Densidad de bordes: {dens_bordes:.1f}%\n"
        f"k_open / k_close: {k_open} / {k_close}\n"
        f"Objetos: {n_objetos}\n"
        f"Área objeto: {area_obj:,} px\n"
        f"Cobertura: {cobertura:.1f}%"
    )
    axes[1, 2].text(0.05, 0.95, texto, fontsize=11, family='monospace',
                    va='top', ha='left', transform=axes[1, 2].transAxes)

    axes[1, 3].hist(V.ravel(), bins=256, color='steelblue', alpha=0.8)
    axes[1, 3].set_title('Histograma V', fontsize=10)
    axes[1, 3].set_xlim(0, 255)

    plt.suptitle(f'{clase_analizar} — {img_path.name}', fontsize=12)
    plt.tight_layout()
    plt.show()

In [ ]:
SPLITS = ['train', 'val', 'test']
BASE_PROCESSED = Path('../data/processed')
BASE_SEGMENTED = Path('../data/segmented')

UMBRAL_DENSIDAD = 0.15
K_GAUSS = 11
EXT = {'.jpg', '.jpeg', '.png'}

for split in SPLITS:
    split_dir = BASE_PROCESSED / split
    if not split_dir.exists():
        print(f'Split no encontrado, se omite: {split_dir}')
        continue

    clases = sorted(c.name for c in split_dir.iterdir() if c.is_dir())

    for clase in clases:
        in_dir = split_dir / clase
        out_dir = BASE_SEGMENTED / split / clase
        out_dir.mkdir(parents=True, exist_ok=True)

        imagenes = sorted(p for p in in_dir.iterdir() if p.suffix.lower() in EXT)
        ok = 0

        for img_path in imagenes:
            img_bgr = cv2.imread(str(img_path))
            if img_bgr is None:
                print(f'  No se pudo leer: {img_path.name}')
                continue
            img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

            hsv_raw = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
            S_raw = cv2.GaussianBlur(hsv_raw[:, :, 1], (5, 5), 0)
            densidad = densidad_bordes(S_raw, K_GAUSS)

            if densidad > UMBRAL_DENSIDAD:
                img_lp = cv2.bilateralFilter(img_bgr, d=50, sigmaColor=220, sigmaSpace=300)
            else:
                img_lp = img_bgr.copy()

            img_hsv = cv2.cvtColor(img_lp, cv2.COLOR_BGR2HSV)
            H, S, V = cv2.split(img_hsv)

            sobel_comb = bordes_sv(S, V, K_GAUSS)
            abierto, cerrado, relleno, k_open, k_close = rellenar_contornos(sobel_comb)
            objeto = cv2.bitwise_and(img_rgb, img_rgb, mask=relleno)

            # guardar: mascara binaria y objeto recortado (de vuelta a BGR)
            cv2.imwrite(str(out_dir / f'{img_path.stem}_mask.png'), relleno)
            cv2.imwrite(str(out_dir / f'{img_path.stem}_obj.png'),
                        cv2.cvtColor(objeto, cv2.COLOR_RGB2BGR))
            ok += 1

        print(f'{split}/{clase}: {ok}/{len(imagenes)} procesadas -> {out_dir}')

print('\nListo. Mascaras y objetos guardados en', BASE_SEGMENTED)